# nb_03a — Gold: dimensions (SCD1 + two flavours of SCD Type 2)

**Module 3 (dimensions).**

| Dimension | Technique | Why |
|---|---|---|
| `dim_date` | generated | date spine |
| `dim_cost_center` | SCD1 (overwrite) | cost centres are stable; carries the RLS region |
| `dim_pay_band` | **SCD2 from a history feed** | grids re-benchmarked yearly; full history available |
| `dim_worker` | **SCD2 via periodic-snapshot MERGE** | staffing actions change worker records |

Every dimension gets an integer **surrogate key** so the fact can resolve the
*version in force on the event date* (nb_03b).

> **Lab notebook.** This is the fill-in-the-blank companion to the solution notebook of the same name. Each code cell only contains `# TODO` comments — use the markdown cell above each one to figure out what to build.


## Imports and the Gold schema

**Summary.** Loads the PySpark and Delta helpers and creates the `gold` schema the dimensions are written into.


In [ ]:
# TODO: Import the helpers this notebook needs and create the gold schema.
# - Import PySpark functions/Window (commonly F / W) and delta.tables.DeltaTable (needed
#   later for the dim_worker MERGE).
# - Create the `gold` schema if it doesn't exist.


## 1. `dim_date`


## `dim_date` — the date spine

**Summary.** Generates a continuous daily calendar with the keys and attributes (year, fiscal year, quarter, month) that power time intelligence in the model.


In [ ]:
# TODO: Build dim_date, a continuous daily calendar spine.
# - Generate one row per day across the lab's date range (2021-01-01..2025-12-31).
# - Derive an integer date_key (yyyyMMdd surrogate key).
# - Derive year, fiscal_year (fiscal year starts in April), quarter, month, and
#   month_name attributes.
# - Overwrite gold.dim_date and print the row count.


## 2. `dim_cost_center` — SCD Type 1

Overwrite current state; add a surrogate key. Carries `hr_region`, the row-level
security driver used in Module 6.


## `dim_cost_center` — SCD Type 1

**Summary.** Overwrites the current cost-centre state and adds a surrogate key; carries `hr_region`, the row-level-security driver used in Module 6.


In [ ]:
# TODO: Build dim_cost_center as a Type-1 (overwrite) dimension.
# - Read bronze.cost_centers.
# - Assign a stable integer cost_center_key (e.g. row_number ordered by cost_center_id).
# - Keep the key plus descriptive columns, including hr_region (used for RLS later).
# - Overwrite gold.dim_cost_center and print the row count.


## 3. `dim_pay_band` — SCD Type 2 from a history feed

Bronze holds one row per (group, level, effective_date). Derive validity per
(group, level) ordered by date: `effective_to` = day before the next version;
`is_current` on the latest. This is the **history-feed → SCD2** pattern.


## `dim_pay_band` — SCD Type 2 from a history feed

**Summary.** Turns the per-effective-date band history into validity ranges: each version's `effective_to` is the day before the next version, and the latest version is flagged current.


In [ ]:
# TODO: Build dim_pay_band as SCD Type 2 from the band-history feed.
# - Order each (classification_group, classification_level)'s versions by
#   band_effective_date using a window function.
# - Set effective_from to the version's date; look ahead to the next version's date to
#   compute effective_to (day before the next version, or 9999-12-31 if there is none).
# - Flag is_current for the open-ended latest version.
# - Assign a surrogate pay_band_key and overwrite gold.dim_pay_band.
# - Print the row count and preview one classification's version history.


## 4. `dim_worker` — SCD Type 2 via periodic-snapshot MERGE

The canonical **close-then-insert** MERGE, tracking `classification_group`,
`classification_level`, `directorate`, `employment_type` via a `row_hash`.

1. **Initial load** — every worker → version 1 (`is_current=true`,
   `effective_to=9999-12-31`).
2. **Second snapshot** (`workers_delta`, effective 2023-07-01) — a MERGE that
   **closes** changed rows and **inserts** the new version; brand-new employees
   are inserted fresh.


## `dim_worker` — SCD Type 2, initial load

**Summary.** Loads the first worker snapshot as version 1 of every worker, hashing the tracked attributes so later snapshots can detect changes.


In [ ]:
# TODO: Load the first worker snapshot as version 1 of dim_worker.
# - Write a small hashing helper that combines the tracked attributes
#   (classification_group, classification_level, directorate, employment_type) into a
#   single row_hash used later to detect changes.
# - Read bronze.workers, cast classification_level, compute row_hash, and set
#   effective_from from the snapshot date, effective_to = 9999-12-31, is_current = true.
# - Overwrite gold.dim_worker with this initial version-1 load and print the row count.


## `dim_worker` — apply the second snapshot (close-then-insert MERGE)

**Summary.** Applies the later `workers_delta` snapshot: the MERGE closes changed current rows, then changed and brand-new workers are inserted as new current versions.


In [ ]:
# TODO: Apply the second worker snapshot with a close-then-insert SCD2 MERGE.
# - Compute row_hash and effective_from for the second snapshot (bronze.workers_delta)
#   the same way as the initial load.
# - Step 1: MERGE into gold.dim_worker matching on employee_id AND is_current = true;
#   when the row_hash differs, close the old version (is_current = false,
#   effective_to = day before the new snapshot's effective_from).
# - Step 2: find workers that are brand new or whose row_hash changed compared to the
#   still-current rows, and append them as new current versions
#   (effective_to = 9999-12-31, is_current = true).
# - Print the current vs. historical row counts to confirm the split.


## `dim_worker` — assign surrogate keys

**Summary.** Adds a stable integer `worker_key` across all worker versions so the fact can resolve the exact version in force on an event date.


In [ ]:
# TODO: Assign a stable surrogate worker_key across all worker versions.
# - Use a window function (e.g. row_number ordered by employee_id, effective_from) so
#   every version of every worker gets a deterministic key.
# - Overwrite gold.dim_worker with the keyed result.
# - Preview a few employees that have more than one version, ordered by effective_from.
